# NeuralSpace Colab Logging Template

Use this notebook to send Colab run metrics and runtime log messages back to NeuralSpace. Generate a one-time claim code in NeuralSpace, then run each cell in order.

## 1. Imports and API base

Update `DEFAULT_API_BASE` only when NeuralSpace gives you a different public API URL.

In [ ]:
from getpass import getpass
from urllib.parse import urlparse
import socket
import time
import requests


DEFAULT_API_BASE = "https://uniform-ate-seo-statistical.trycloudflare.com/api/v1"

## 2. Define NeuralSpace logging client

In [ ]:
class NeuralSpaceLogger:
    def __init__(self, api_base, runtime_token, bootstrap):
        self.api_base = api_base.rstrip("/")
        self.headers = {"Authorization": f"Bearer {runtime_token}"}
        self.session_id = bootstrap["session_id"]
        self.run_id = None

    @classmethod
    def connect(cls, api_base=None):
        api_base = (api_base or DEFAULT_API_BASE).strip().rstrip("/")
        print(f"Using NeuralSpace API: {api_base}")
        if not (api_base.startswith("https://") or api_base.startswith("http://localhost") or api_base.startswith("http://127.0.0.1")):
            raise RuntimeError("Use HTTPS for remote NeuralSpace APIs")
        host = urlparse(api_base).hostname
        try:
            socket.gethostbyname(host)
        except OSError as exc:
            raise RuntimeError(
                f"Cannot resolve NeuralSpace API host: {host}. "
                "If you are using trycloudflare.com, restart the Cloudflare tunnel and paste the fresh public URL."
            ) from exc
        claim_code = getpass("Paste NeuralSpace claim code: ").strip()
        response = requests.post(f"{api_base}/colab/claims/exchange", json={"claim_code": claim_code}, timeout=30)
        response.raise_for_status()
        bootstrap = response.json()
        return cls(api_base, bootstrap["runtime_token"], bootstrap)

    def request(self, method, path, **kwargs):
        response = requests.request(method, f"{self.api_base}{path}", headers=self.headers, timeout=kwargs.pop("timeout", 30), **kwargs)
        response.raise_for_status()
        if response.content:
            return response.json()
        return None

    def heartbeat(self):
        return self.request("POST", "/colab/runtime/heartbeat")

    def start_run(self, name="Colab training run"):
        run = self.request("POST", "/colab/runtime/runs", json={"name": name})
        self.run_id = run["run_id"]
        return run

    def ensure_run(self):
        if not self.run_id:
            self.start_run()
        return self.run_id

    def log_metrics(self, values, step=None):
        run_id = self.ensure_run()
        payload = {"values": values}
        if step is not None:
            payload["step"] = step
        return self.request("POST", f"/colab/runtime/runs/{run_id}/metrics", json=payload)

    def log(self, message, level="INFO"):
        run_id = self.ensure_run()
        return self.request(
            "POST",
            f"/colab/runtime/runs/{run_id}/logs",
            json={"level": level, "message": message},
        )

## 3. Connect to NeuralSpace

In [ ]:
logger = NeuralSpaceLogger.connect()
logger.heartbeat()
print(f"Connected to NeuralSpace session {logger.session_id}")

## 4. Create a run

In [ ]:
run = logger.start_run("Colab logging template")
print(f"Run created: {run['run_id']}")

## 5. Log metrics

Replace the simulated values with metrics from your own model training code.

In [ ]:
for epoch in range(1, 6):
    loss = round(1.0 / epoch, 4)
    accuracy = round(0.75 + epoch * 0.04, 4)
    logger.log_metrics({"loss": loss, "accuracy": accuracy}, step=epoch)
    time.sleep(0.5)

print("Metrics sent to NeuralSpace")

## 6. Log runtime messages

Use levels such as `INFO`, `WARNING`, or `ERROR` to make runtime events easier to scan in NeuralSpace.

In [ ]:
logger.log("Training started")
logger.log("Validation accuracy is below target threshold", level="WARNING")
logger.log("Training finished successfully")
print("Logs sent to NeuralSpace")